In [1]:
#this was a workaround to make torch import and run on my computer
import os 
os.environ['KMP_DUPLICATE_LIB_OK'] = "TRUE"
import torch

In [2]:
import matplotlib.pyplot as plt

In [3]:
import torch.nn.functional as F

In [4]:
file = open("input.txt" , 'r')
content = file.read()
sentences = content.splitlines()

all_words = []

for sentence in sentences:
    words = sentence.split()
    all_words.extend(words)

unique_words = sorted(list(set(all_words)))

In [5]:
#creating the mapping 
stoi = {s:i+1 for i , s in enumerate(unique_words)}
stoi["."] = 0#this is the special starting token marking the start and the end 
itos ={i:s for s, i in stoi.items()}
vocab_size = len(itos)
vocab_size

25671

In [6]:
len(sentences)
len(unique_words)

25670

In [7]:
def build_dataset(sentences):
    X , Y=  [] , []
    context_window = 10

    for sentence in sentences:
        words =sentence.split()
        context=  [0]*context_window

        for word in words+["."]:
            ix = stoi[word]
            X.append(context)
            Y.append(ix)
            context = context[1:]+[ix]
            #yes - it has rolling context ie 1 ,2 , 3 - > 4; 2 ,3 , 4-> 5----> this NPLM model is also  an  n gram in its core but just with embeddings
            # and way larger and alterable context


    X = torch.tensor(X)
    Y = torch.tensor(Y)

    print(X.shape , Y.shape)
    return X , Y 


In [8]:
#dividing the data into 3 segments the train data , the val data and the test data
# the train data would be 80% of the whole dataset ; the val data would be 10% and the test data would be 10%
import random 
random.seed(3801)
random.shuffle(sentences)

n1 = int(0.8*(len(sentences)))
n2= int(0.9*(len(sentences)))

Xtr , Ytr = build_dataset(sentences[:n1])
Xval , Yval = build_dataset(sentences[n1:n2])
Xtest , Ytest = build_dataset(sentences[n2:])

torch.Size([194521, 10]) torch.Size([194521])
torch.Size([24075, 10]) torch.Size([24075])
torch.Size([24055, 10]) torch.Size([24055])


In [31]:
#parameters 
g = torch.Generator().manual_seed(3801)
n_emb = 100
context_window = 10
hidden_neuron = 200
C = torch.randn((vocab_size , n_emb) ,generator  =g)
W1 = torch.randn((n_emb*context_window , hidden_neuron) , generator = g)*0.01
b1 = torch.randn((hidden_neuron) ,  generator = g)
W2 = torch.randn((hidden_neuron , vocab_size) , generator = g)*0.001
b2 = torch.randn((vocab_size) , generator = g)*1

parameters = [C , W1 , W2 , b1 , b2]
sum(p.nelement() for p in parameters)
wd  = 1e-4

for p in parameters:
    p.requires_grad = True

In [32]:
#find out the best learning rate 
lre =torch.linspace(-3 ,0 , 1000)
lrs = 10**lre

In [36]:
#forward pass 
max_steps = 10000
batch_size = 200 #so i am implementing the entire training in batch sizes - so as for better performance 
#and less load on the CPU
lossi = []#this would have some elements which it would contain - to say graph the entire thingy

for i in range(max_steps):
    #the minibatch construction
    ix = torch.randint(0 , Xtr.shape[0] , (batch_size , ) , generator = g )
    Xb , Yb = Xtr[ix] , Ytr[ix]

    #implementing the forward pass
    emb = C[Xb]
    embcat = emb.view(batch_size , context_window*n_emb)
    h = torch.tanh(embcat@W1 + b1)
    logits = h@W2+b2 
    loss = F.cross_entropy(logits , Yb)
    perplexity = loss.exp()


    #implementing the backward pass 
    for p in parameters:
        p.grad = None
    loss.backward()

    #the data update - trying to figure out the loss and the learning rate
    lr = 10**(-0.1)
    for p in parameters:
        p.data += -lr*p.grad - wd*p.data
    lossi.append(loss.item())

    if i%1000==0:
        print(i)
    

print(loss.item())
print(perplexity.item())

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
4.850419521331787
127.79399108886719


In [39]:
@torch.no_grad()#this decorator disables gradient tracking 
def split_loss(split):
    x , y = {

    'train' : (Xtr , Ytr) ,
    'val': (Xval , Yval),
    'test' : (Xtest , Ytest),
        
    }[split]
    batch_size = 200
    ix = torch.randint(0 , x.shape[0] , (batch_size , ) , generator = g )
    Xb , Yb = x[ix] , y[ix]
    emb = C[Xb]
    embcat = emb.view(emb.shape[0], -1)
    h = torch.tanh(embcat@W1 + b1)
    h = F.dropout(h , p = 0.3 , training = False)
    logits = h@W2+b2 
    loss = F.cross_entropy(logits , y[ix])
    perplexity = loss.exp()
    print(split , loss.item())


split_loss('train')
split_loss('val')

train 4.4743242263793945
val 6.393610000610352


In [40]:
#sample from the model
g  = torch.Generator().manual_seed(2407)


for _ in range(20):
    out = []
    context_window =  10
    context = [0]*context_window

    while True:
        emb = C[torch.tensor([context])]
        embcat = emb.view( 1,-1)
        h = torch.tanh(embcat@W1 + b1)
        h = F.dropout(h , p = 0.3 , training = False)
        logits = h@W2+b2 
        probs = F.softmax(logits , dim  = 1)
        ix = torch.multinomial(probs , num_samples = 1 ,  generator = g).item()
        context = context[1:]+[ix]
        out.append(ix)
        if ix == 0 :
            break 

    print(" ".join(itos[i] for i in out))

        

.
PRINCE EDWARD: .
.
.
.
.
.
My granted: Falconbridge torment in my lip, religion. .
.
shriek'd .
QUEEN MARGARET: .
.
.
.
That Henry, and rash: drops 'if pause, .
.
.
.
.
.
